In [5]:
import pandas as pd
import os
from pathlib import Path

# ------------------------------------------------------------
# STEP 1: Get all CSV files from analysis_data folder
# ------------------------------------------------------------
folder_path = "analysis_data"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {f}")

# ------------------------------------------------------------
# STEP 2: Read each CSV and create a dataframe
# ------------------------------------------------------------
dfs_to_merge = []

for csv_file in csv_files:
    # Read the CSV
    file_path = os.path.join(folder_path, csv_file)
    df = pd.read_csv(file_path)
    
    # Get column name from filename (remove .csv extension)
    column_name = Path(csv_file).stem
    
    # Rename 'score' column to the filename
    df = df.rename(columns={"score": column_name})
    
    # Keep only group_name and the score column
    df = df[["group_name", column_name]]
    
    dfs_to_merge.append(df)

# ------------------------------------------------------------
# STEP 3: Merge all dataframes on group_name
# ------------------------------------------------------------
# Start with the first dataframe
result_df = dfs_to_merge[0]

# Merge each subsequent dataframe
for df in dfs_to_merge[1:]:
    result_df = result_df.merge(df, on="group_name", how="outer")

# Sort by group_name for easier reading
result_df = result_df.sort_values("group_name").reset_index(drop=True)

# Fill any missing values with 0
result_df = result_df.fillna(0)

# ------------------------------------------------------------
# STEP 4: Create final_score column (sum across all scores)
# ------------------------------------------------------------
# Get all columns except group_name
score_columns = [col for col in result_df.columns if col != "group_name"]

# Sum across all score columns
result_df["final_score"] = result_df[score_columns].sum(axis=1)

# Sort by final_score (highest to lowest)
result_df = result_df.sort_values("final_score", ascending=False).reset_index(drop=True)

result_df

Found 4 CSV files:
  - technique_defense.csv
  - technique_diversity.csv
  - technique_novelty.csv
  - technique_predictions.csv


,group_name,technique_defense,technique_diversity,technique_novelty,technique_predictions,final_score
0,Kimsuky,0.88,0.990614,0.250318,1.000000,3.120932
1,Lazarus Group,0.96,0.989905,0.230398,0.858381,3.038683
2,Volt Typhoon,1.00,0.865946,0.292678,0.753235,2.911859
3,APT28,0.56,0.995620,0.360630,0.835831,2.752080
4,Mustang Panda,0.72,0.930103,0.318380,0.776092,2.744575
...,...,...,...,...,...,...
163,TA459,0.00,0.215564,0.008275,0.035890,0.259730
164,APT17,0.00,0.000000,0.210326,0.017728,0.228054
165,Moafee,0.00,0.000000,0.218411,0.006783,0.225194
166,APT16,0.00,0.000000,0.169898,0.007604,0.177503
